In [60]:
# Import necessary libraries for data analysis and association rule mining
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from data_profiling import ProfileReport
import numpy as np

# 4.1. Load and combine the data

In [61]:
#Loading the datasets
marvel_raw = pd.read_csv('../datasets/marvel-wikia-data.csv')
dc_raw = pd.read_csv('../datasets/dc-wikia-data.csv')

In [62]:
marvel_raw.head()

,page_id,name,urlslug,ID,ALIGN,EYE,HAIR,SEX,GSM,ALIVE,APPEARANCES,FIRST APPEARANCE,Year
0,1678,Spider-Man (Peter Parker),\/Spider-Man_(Peter_Parker),Secret Identity,Good Characters,Hazel Eyes,Brown Hair,Male Characters,NaN,Living Characters,4043.0,Aug-62,1962.0
1,7139,Captain America (Steven Rogers),\/Captain_America_(Steven_Rogers),Public Identity,Good Characters,Blue Eyes,White Hair,Male Characters,NaN,Living Characters,3360.0,Mar-41,1941.0
2,64786,"Wolverine (James \""Logan\"" Howlett)",\/Wolverine_(James_%22Logan%22_Howlett),Public Identity,Neutral Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,3061.0,Oct-74,1974.0
3,1868,"Iron Man (Anthony \""Tony\"" Stark)",\/Iron_Man_(Anthony_%22Tony%22_Stark),Public Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,2961.0,Mar-63,1963.0
4,2460,Thor (Thor Odinson),\/Thor_(Thor_Odinson),No Dual Identity,Good Characters,Blue Eyes,Blond Hair,Male Characters,NaN,Living Characters,2258.0,Nov-50,1950.0


In [63]:
dc_raw.head()

,page_id,name,urlslug,ID,ALIGN,EYE,HAIR,SEX,GSM,ALIVE,APPEARANCES,FIRST APPEARANCE,YEAR
0,1422,Batman (Bruce Wayne),\/wiki\/Batman_(Bruce_Wayne),Secret Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,3093.0,"1939, May",1939.0
1,23387,Superman (Clark Kent),\/wiki\/Superman_(Clark_Kent),Secret Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,2496.0,"1986, October",1986.0
2,1458,Green Lantern (Hal Jordan),\/wiki\/Green_Lantern_(Hal_Jordan),Secret Identity,Good Characters,Brown Eyes,Brown Hair,Male Characters,NaN,Living Characters,1565.0,"1959, October",1959.0
3,1659,James Gordon (New Earth),\/wiki\/James_Gordon_(New_Earth),Public Identity,Good Characters,Brown Eyes,White Hair,Male Characters,NaN,Living Characters,1316.0,"1987, February",1987.0
4,1576,Richard Grayson (New Earth),\/wiki\/Richard_Grayson_(New_Earth),Secret Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,1237.0,"1940, April",1940.0


In [64]:
# Creating copies of the raw datasets
marvel = marvel_raw.copy()
dc = dc_raw.copy()

In [65]:
# Adding a PUBLISHER column to each dataset

marvel['PUBLISHER'] = 'Marvel'
dc['PUBLISHER'] = 'DC'

In [66]:
# Standardizing column names and comparing them between Marvel and DC datasets

marvel.columns = marvel.columns.str.strip().str.lower().str.replace(' ', '_')
dc.columns = dc.columns.str.strip().str.lower().str.replace(' ', '_')

for i in range(len(marvel.columns)):
    if marvel.columns[i] == dc.columns[i]:
        print(f'{dc.columns[i]} = {marvel.columns[i]}')
    else:
        print('different')

page_id = page_id
name = name
urlslug = urlslug
id = id
align = align
eye = eye
hair = hair
sex = sex
gsm = gsm
alive = alive
appearances = appearances
first_appearance = first_appearance
year = year
publisher = publisher


In [67]:
comics = pd.concat([marvel, dc], ignore_index=True)
comics.head()

,page_id,name,urlslug,id,align,eye,hair,sex,gsm,alive,appearances,first_appearance,year,publisher
0,1678,Spider-Man (Peter Parker),\/Spider-Man_(Peter_Parker),Secret Identity,Good Characters,Hazel Eyes,Brown Hair,Male Characters,NaN,Living Characters,4043.0,Aug-62,1962.0,Marvel
1,7139,Captain America (Steven Rogers),\/Captain_America_(Steven_Rogers),Public Identity,Good Characters,Blue Eyes,White Hair,Male Characters,NaN,Living Characters,3360.0,Mar-41,1941.0,Marvel
2,64786,"Wolverine (James \""Logan\"" Howlett)",\/Wolverine_(James_%22Logan%22_Howlett),Public Identity,Neutral Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,3061.0,Oct-74,1974.0,Marvel
3,1868,"Iron Man (Anthony \""Tony\"" Stark)",\/Iron_Man_(Anthony_%22Tony%22_Stark),Public Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,2961.0,Mar-63,1963.0,Marvel
4,2460,Thor (Thor Odinson),\/Thor_(Thor_Odinson),No Dual Identity,Good Characters,Blue Eyes,Blond Hair,Male Characters,NaN,Living Characters,2258.0,Nov-50,1950.0,Marvel


# 4.2. Examine the data

## 4.2 (a)

In [68]:
missing_values = pd.DataFrame({
    "Missing Count": comics.isna().sum(),
    "Missing Percentage (%)": (comics.isna().sum() / len(comics) * 100).round(2)
})

missing_values


,Missing Count,Missing Percentage (%)
page_id,0,0.00
name,0,0.00
urlslug,0,0.00
id,5783,24.85
align,3413,14.67
eye,13395,57.56
hair,6538,28.09
sex,979,4.21
gsm,23118,99.34
alive,6,0.03


In [69]:
print("Duplicated rows (full record):", comics.duplicated().sum())
print("Duplicated page_id:", comics.duplicated(subset=["page_id"]).sum())
print("Duplicated publisher and page_id:",comics.duplicated(subset=["publisher", "page_id"]).sum())

Duplicated rows (full record): 0
Duplicated page_id: 476
Duplicated publisher and page_id: 0


In [70]:
categorical_columns = comics.select_dtypes(include=["object", "string", "category"]).columns

for column in categorical_columns:
    print(f"\n{column}:")
    print(comics[column].value_counts(dropna=False))


name:
name
Spider-Man (Peter Parker)       1
Primus (Flb'Dbi) (Earth-616)    1
Nyambe (Earth-616)              1
Juber (Shi'ar) (Earth-616)      1
George Clinton (Earth-616)      1
                               ..
Skye (Earth-616)                1
Singer (Earth-616)              1
Shockett (Earth-616)            1
Rick Barron (Earth-616)         1
Mookie (New Earth)              1
Name: count, Length: 23272, dtype: int64

urlslug:
urlslug
\/Spider-Man_(Peter_Parker)         1
\/Primus_(Flb%27Dbi)_(Earth-616)    1
\/Nyambe_(Earth-616)                1
\/Juber_(Shi%27ar)_(Earth-616)      1
\/George_Clinton_(Earth-616)        1
                                   ..
\/Skye_(Earth-616)                  1
\/Singer_(Earth-616)                1
\/Shockett_(Earth-616)              1
\/Rick_Barron_(Earth-616)           1
\/wiki\/Mookie_(New_Earth)          1
Name: count, Length: 23272, dtype: int64

id:
id
Secret Identity                  8683
Public Identity                  6994
NaN         

In [71]:
comics[["appearances", "year"]].describe().T

,count,mean,std,min,25%,50%,75%,max
appearances,21821.0,19.009303,93.814040,1.0,1.0,4.0,10.0,4043.0
year,22388.0,1986.420046,18.972698,1935.0,1976.0,1990.0,2001.0,2013.0


In [72]:
for name, df in [("Marvel", marvel), ("DC", dc)]:
    print(f"\n{name}")

    print("\nFirst appearance examples:")
    print(df["first_appearance"].dropna().head(10))

    print("\nFirst appearance data type:")
    print(df["first_appearance"].dtype)

    print("\nNumeric distributions:")
    print(df[["appearances", "year"]].describe())


Marvel

First appearance examples:
0    Aug-62
1    Mar-41
2    Oct-74
3    Mar-63
4    Nov-50
5    Nov-61
6    Nov-61
7    May-62
8    Sep-63
9    Nov-61
Name: first_appearance, dtype: object

First appearance data type:
object

Numeric distributions:
        appearances          year
count  15280.000000  15561.000000
mean      17.033377   1984.951803
std       96.372959     19.663571
min        1.000000   1939.000000
25%        1.000000   1974.000000
50%        3.000000   1990.000000
75%        8.000000   2000.000000
max     4043.000000   2013.000000

DC

First appearance examples:
0         1939, May
1     1986, October
2     1959, October
3    1987, February
4       1940, April
5    1941, December
6    1941, November
7      1989, August
8    1969, November
9     1956, October
Name: first_appearance, dtype: object

First appearance data type:
object

Numeric distributions:
       appearances         year
count  6541.000000  6827.000000
mean     23.625134  1989.766662
std      87.37

## 4.2(b)

In [73]:
profile = ProfileReport(comics, title="Comics Dataset Profile")
profile.to_file("../outputs/Thirumurugan_Bocardo_part1_comics_EDA.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/adrianbocardosantos/code_projects/programming_for_data_science/project2/project2_env/lib/python3.14/site-packages/data_profiling/model/describe.py:85: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_start = datetime.utcnow()
100%|██████████| 14/14 [00:00<00:00, 211.82it/s]
/Users/adrianbocardosantos/code_projects/programming_for_data_science/project2/project2_env/lib/python3.14/site-packages/data_profiling/model/describe.py:190: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_end = datetime.utcnow()


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## 4.2 (c)

### Q1 List at least four data-quality issues you found in a) and b). For each one, give the column(s), the number of rows affected, and how you will fix it in Section 4.3. This list is your cleaning plan.

1. **`gsm`:** 23,118 rows (99.3%) have missing values. We will drop this column because it contains very little observed information.

2. **Missing categorical values:** We will fill missing item values with `"Unknown"` rather than guess character attributes. This includes `id` (5,783 rows), `align` (3,413), `eye` (13,395), `hair` (6,538), `sex` (979), and `alive` (6). Moving the misplaced `"Auburn Hair"` values changes the `eye` and `hair` missing counts by seven.

3. **`urlslug`:** All 23,272 rows contain a URL path. We will drop this column because it is not needed as an item in the association-rule analysis.

4. **`name`:** Character names identify individual records rather than useful groups for association rules. We will drop this column from the item table. The names remain available in the original `comics` DataFrame.

5. **`first_appearance` and `year`:** Both columns contain year information for 22,388 rows. We will drop `first_appearance`, use `year` to create a categorical comic-book era, and then drop the raw `year` column from the item table.

6. **`eye` and `hair`:** Seven rows have `"Auburn Hair"` in the `eye` column while `hair` is missing. We will move the value to `hair` and mark the eye value as missing. The missing eye value will later be filled with `"Unknown"`.

### Q2 Is page_id a valid key for the combined data? What would happen if you dropped duplicates on page_id?

No, `page_id` alone is not a valid unique key for the combined data. We found **476 repeated `page_id` values** because the same ID can appear in both publishers’ files.

If we dropped duplicates based only on `page_id`, we would remove **476 character records**, even though they are not fully duplicated rows. The combination of `publisher` and `page_id` has no duplicates, so we can use both to create a unique `character_id`.

### Q3 Which of your issues did the EDA report flag (as alerts or in its statistics), and which did you find only with pandas? Why can a profile report miss some problems?

The EDA report flagged missing values in several columns, including `gsm` (99.3% missing), `eye` (57.6%), `hair` (28.1%), `id` (24.8%), and `sex` (4.2%). It also flagged `name` and `urlslug` as having unique values, which helped us identify them as unsuitable items for association-rule analysis.

Using pandas, we checked that `page_id` has 476 repeated values in the combined data and that `publisher` together with `page_id` is unique. We also found seven `"Auburn Hair"` values in the `eye` column and examined the different `first_appearance` formats. These issues were not explicitly flagged as alerts, although some of the underlying values may have appeared in the report’s statistics.

A profile report can miss problems because it summarizes data using general rules. It may not know that `"Auburn Hair"` belongs in `hair`, or that `page_id` is intended to identify one character across two publishers. Finding these issues requires us to inspect the values and understand their meaning.

# 4.3 Clean and transform

In [74]:
month_dict = {'Jan': 'January', 'Feb': 'February', 'Mar': 'March', 'Apr': 'April', 'May': 'May', 'Jun': 'June', 'Jul': 'July', 'Aug': 'August', 'Sep': 'September', 'Oct': 'October', 'Nov': 'November', 'Dec': 'December'}

In [75]:
# Create a cleaned copy dropping columns not needed for the analysis

comics_df = comics.copy()

comics_df["page_id"] = (comics_df["publisher"] + "_" + comics_df["page_id"].astype(str))

comics_df = comics_df.rename(columns={"page_id": "character_id"})

comics_df = comics_df.drop(columns=["first_appearance", "urlslug", "gsm", "name"])

comics_df["character_id"].duplicated().sum()

np.int64(0)

In [76]:
# Move misplaced hair values to hair; the corresponding eye color is unknown
mask = comics_df["eye"] == "Auburn Hair"

comics_df.loc[mask, "hair"] = "Auburn Hair"
comics_df.loc[mask, "eye"] = pd.NA

In [77]:
# Shorten category labels and remove surrounding whitespace
comics_df['id'] = comics_df['id'].str.replace('Identity','').str.strip()
comics_df['align'] = comics_df['align'].str.replace('Characters','').str.strip()
comics_df['hair'] = comics_df['hair'].str.replace('Hair','').str.strip()
comics_df['sex'] = comics_df['sex'].str.replace('Characters','').str.strip()
comics_df['alive'] = comics_df['alive'].str.replace('Characters','').str.strip()
comics_df['eye'] = comics_df['eye'].str.replace('Eyes','').str.strip()

In [ ]:
# Merge equivalent labels
comics_df["eye"] = comics_df["eye"].astype("string").replace({
    "Yellow Eyeballs": "Yellow",
    "Black Eyeballs": "Black"
})

comics_df["hair"] = comics_df["hair"].astype("string").replace({
    "Bald": "No"
})

# Group known categories with fewer than 10 records; keep missing values separate
for column in ["eye", "hair"]:
    counts = comics_df[column].value_counts()
    rare = counts[counts < 10].index
    comics_df[column] = comics_df[column].where(
        ~comics_df[column].isin(rare), "Other"
    )
    comics_df[column] = comics_df[column].astype("category")

'''We applied the threshold only to the eye and hair columns because the other categorical columns have relatively few categories. 
We grouped values with fewer than 10 records into "Other" to simplify the analysis while keeping more common 
categories separate. We applied the threshold after merging equivalent labels and kept missing values separate from "Other".'''

,character_id,id,align,eye,hair,sex,alive,appearances,year,publisher
0,Marvel_1678,Secret,Good,Hazel,Brown,Male,Living,4043.0,1962.0,Marvel
1,Marvel_7139,Public,Good,Blue,White,Male,Living,3360.0,1941.0,Marvel
2,Marvel_64786,Public,Neutral,Blue,Black,Male,Living,3061.0,1974.0,Marvel
3,Marvel_1868,Public,Good,Blue,Black,Male,Living,2961.0,1963.0,Marvel
4,Marvel_2460,No Dual,Good,Blue,Blond,Male,Living,2258.0,1950.0,Marvel
...,...,...,...,...,...,...,...,...,...,...
23267,DC_66302,Public,Good,<NA>,<NA>,Female,Living,NaN,NaN,DC
23268,DC_283475,Public,Good,<NA>,<NA>,Male,Living,NaN,NaN,DC
23269,DC_283478,Public,Good,<NA>,<NA>,Male,Living,NaN,NaN,DC
23270,DC_283471,Public,Good,<NA>,<NA>,Male,Living,NaN,NaN,DC


In [79]:
comics_df['year'] = comics_df['year'].astype("Int64")
comics_df["year_group"] = pd.cut(comics_df["year"],bins=[1935, 1955, 1969, 1984, 2013 ], labels=["Golden", "Silver", "Bronze", "Modern"], include_lowest=True)
comics_df["appearance_bin"] = pd.cut(comics_df["appearances"],bins=[0, 1, 5, 10, 50, 100, float("inf")],labels=["Single appearance","Very few","Few","Moderate","Many","Very many"], include_lowest=True)

comics_df= comics_df.drop(columns=["year", "appearances"])
comics_df

,character_id,id,align,eye,hair,sex,alive,publisher,year_group,appearance_bin
0,Marvel_1678,Secret,Good,Hazel,Brown,Male,Living,Marvel,Silver,Very many
1,Marvel_7139,Public,Good,Blue,White,Male,Living,Marvel,Golden,Very many
2,Marvel_64786,Public,Neutral,Blue,Black,Male,Living,Marvel,Bronze,Very many
3,Marvel_1868,Public,Good,Blue,Black,Male,Living,Marvel,Silver,Very many
4,Marvel_2460,No Dual,Good,Blue,Blond,Male,Living,Marvel,Golden,Very many
...,...,...,...,...,...,...,...,...,...,...
23267,DC_66302,Public,Good,<NA>,<NA>,Female,Living,DC,NaN,NaN
23268,DC_283475,Public,Good,<NA>,<NA>,Male,Living,DC,NaN,NaN
23269,DC_283478,Public,Good,<NA>,<NA>,Male,Living,DC,NaN,NaN
23270,DC_283471,Public,Good,<NA>,<NA>,Male,Living,DC,NaN,NaN


We retained characters with missing values and created an "Unknown" item for each missing variable. This preserves all characters and allows us to examine patterns in missing information without guessing their attributes.

In [81]:
item_columns = [
    "publisher", "id", "align", "eye", "hair",
    "sex", "alive", "year_group", "appearance_bin"
]

for column in item_columns:
    comics_df[column] = comics_df[column].astype("string").fillna("Unknown")

# Store the categorical columns using pandas category dtype
categorical_columns = ["id", "align", "eye", "hair", "sex", "alive", "publisher", 'year_group', 'appearance_bin']

comics_df[categorical_columns] = comics_df[categorical_columns].astype("category")


comics_df

,character_id,id,align,eye,hair,sex,alive,publisher,year_group,appearance_bin
0,Marvel_1678,Secret,Good,Hazel,Brown,Male,Living,Marvel,Silver,Very many
1,Marvel_7139,Public,Good,Blue,White,Male,Living,Marvel,Golden,Very many
2,Marvel_64786,Public,Neutral,Blue,Black,Male,Living,Marvel,Bronze,Very many
3,Marvel_1868,Public,Good,Blue,Black,Male,Living,Marvel,Silver,Very many
4,Marvel_2460,No Dual,Good,Blue,Blond,Male,Living,Marvel,Golden,Very many
...,...,...,...,...,...,...,...,...,...,...
23267,DC_66302,Public,Good,Unknown,Unknown,Female,Living,DC,Unknown,Unknown
23268,DC_283475,Public,Good,Unknown,Unknown,Male,Living,DC,Unknown,Unknown
23269,DC_283478,Public,Good,Unknown,Unknown,Male,Living,DC,Unknown,Unknown
23270,DC_283471,Public,Good,Unknown,Unknown,Male,Living,DC,Unknown,Unknown


In [82]:
#Builiding the transaction matrix
basket = pd.get_dummies(comics_df.set_index('character_id'), prefix_sep='=')
basket

,id=Known to Authorities,id=No Dual,id=Public,id=Secret,id=Unknown,align=Bad,align=Good,align=Neutral,align=Reformed Criminals,align=Unknown,...,year_group=Modern,year_group=Silver,year_group=Unknown,appearance_bin=Few,appearance_bin=Many,appearance_bin=Moderate,appearance_bin=Single appearance,appearance_bin=Unknown,appearance_bin=Very few,appearance_bin=Very many
character_id,,,,,,,,,,,,,,,,,,,,,
Marvel_1678,False,False,False,True,False,False,True,False,False,False,...,False,True,False,False,False,False,False,False,False,True
Marvel_7139,False,False,True,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
Marvel_64786,False,False,True,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,True
Marvel_1868,False,False,True,False,False,False,True,False,False,False,...,False,True,False,False,False,False,False,False,False,True
Marvel_2460,False,True,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DC_66302,False,False,True,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,False,False
DC_283475,False,False,True,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,False,False
DC_283478,False,False,True,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,False,False


In [89]:
long_item = comics_df.melt(id_vars="character_id").set_index("character_id")

long_item["item"] = (
    long_item["variable"] + "=" + long_item["value"].astype("string")
)

long_item = long_item["item"]
long_item

character_id
Marvel_1678                  id=Secret
Marvel_7139                  id=Public
Marvel_64786                 id=Public
Marvel_1868                  id=Public
Marvel_2460                 id=No Dual
                         ...          
DC_66302        appearance_bin=Unknown
DC_283475       appearance_bin=Unknown
DC_283478       appearance_bin=Unknown
DC_283471       appearance_bin=Unknown
DC_150660       appearance_bin=Unknown
Name: item, Length: 209448, dtype: string

In [105]:
rebuilt_basket = pd.crosstab(long_item.index, long_item).astype(bool)
rebuilt_basket.index.name = "character_id"
rebuilt_basket

item,align=Bad,align=Good,align=Neutral,align=Reformed Criminals,align=Unknown,alive=Deceased,alive=Living,alive=Unknown,appearance_bin=Few,appearance_bin=Many,...,sex=Genderfluid,sex=Genderless,sex=Male,sex=Transgender,sex=Unknown,year_group=Bronze,year_group=Golden,year_group=Modern,year_group=Silver,year_group=Unknown
character_id,,,,,,,,,,,,,,,,,,,,,
DC_10007,False,True,False,False,False,False,True,False,True,False,...,False,False,True,False,False,False,False,True,False,False
DC_100156,True,False,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
DC_100159,True,False,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,False,False
DC_100164,False,True,False,False,False,True,False,False,True,False,...,False,False,True,False,False,False,False,True,False,False
DC_100176,False,True,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Marvel_9954,False,True,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
Marvel_9967,True,False,False,False,False,False,True,False,False,False,...,False,False,True,False,False,True,False,False,False,False
Marvel_99799,False,True,False,False,False,True,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False


# 4.4. Mine the rules